# Anti-bot Mechanisms Analysis

This notebook helps you to:

1. Load CSV with benchmarking results.
2. Display figures and precise number not included in the paper.
3. ReproduiceTable 7 in appendix D.
4. Reproduice table 3 of section 5.

In [1]:
import pandas as pd

# Better dataframe display
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

In [2]:
CSV_PATH = "./tests_anti-bot_mechanisms.csv"

df = pd.read_csv(CSV_PATH)

print(df.shape)
df.head()

(1449, 25)


,Test Index,Web Agent,Agent LOCAL/CLOUD,Agent Version,Model,Model Local/Cloud,Agent Browser,Browser Local/Cloud,Prompt Number,Special prompt,Website,Time,Date,BLOCKED,TIMEOUT,sessionId,First Name,Last Name,Post Content,Defense,Cookie ID,Selenium - Puppeteer - Playwright,test_cookieId,siteKey,datetime
0,115,Antropic Claude for Chrome,LOCAL,1.0.40,claude-sonnet-4.5,CLOUD,Google Chrome 144.0.7559.96,LOCAL,1,False,https://example.org/,13:33:00,28/01/2026,False,False,61zxper9,Meridian,Silverbrook,"Time flows like a river through the mountains,...",NONE,NONE,NONE,NONE,site0,2026-01-28 12:33:00+00:00
1,116,Antropic Claude for Chrome,LOCAL,1.0.40,claude-sonnet-4.5,CLOUD,Google Chrome 144.0.7559.96,LOCAL,2,False,https://example.org/,13:35:00,28/01/2026,False,False,1o4d8npg,Evelyn,Harrington,The gentle rhythm of a clock's mechanism mirro...,NONE,NONE,NONE,NONE,site0,2026-01-28 12:35:00+00:00
2,117,Antropic Claude for Chrome,LOCAL,1.0.40,claude-sonnet-4.5,CLOUD,Google Chrome 144.0.7559.96,LOCAL,3,False,https://example.org/,13:37:00,28/01/2026,False,False,7ee434p4,Cassian,Riverwood,The intricate gears within timepieces represen...,NONE,NONE,NONE,NONE,site0,2026-01-28 12:37:00+00:00
3,118,Antropic Claude for Chrome,LOCAL,1.0.40,claude-sonnet-4.5,CLOUD,Google Chrome 144.0.7559.96,LOCAL,4,False,https://example.org/,13:39:00,28/01/2026,False,False,1h74frfp,Damaris,Valebrook,Beneath the ancient towers where time began it...,NONE,NONE,NONE,NONE,site0,2026-01-28 12:39:00+00:00
4,119,Antropic Claude for Chrome,LOCAL,1.0.40,claude-sonnet-4.5,CLOUD,Google Chrome 144.0.7559.96,LOCAL,5,False,https://example.org/,13:43:00,28/01/2026,False,False,1mmti5zv,Artemis,Thornfield,The symphony of ticking mechanisms reminds us ...,NONE,NONE,NONE,NONE,site0,2026-01-28 12:43:00+00:00


In [3]:
def build_agent_label(row):

    web_agent = str(row["Web Agent"]).lower()
    model = str(row["Model"]).lower()
    location = str(row["Agent LOCAL/CLOUD"]).upper()

    # Humans
    if "human" in web_agent:
        return "human"

    # CLI tools
    if "curl" in web_agent:
        return "curl"

    if "wget" in web_agent:
        return "wget"

    if "scrapy" in web_agent:
        return "scrapy"

    # Browsers
    if "selenium" in web_agent:
        return "selenium"

    if "playwright" in web_agent:
        return "playwright"

    if "puppeteer" in web_agent:
        return "puppeteer"

    # OpenClaw
    if "openclaw" in web_agent and "sonnet" in model:
        return "openclaw sonnet"

    if "openclaw" in web_agent and "opus" in model:
        return "openclaw opus"

    # Claude Chrome
    if "chrome" in web_agent and "sonnet" in model:
        return "claude chrome sonnet"

    if "chrome" in web_agent and "opus" in model:
        return "claude chrome opus"

    # Crawl4AI
    if "crawl4ai_undetected" in web_agent:
        return "crawl4ai-underetected"

    if "crawl4ai_stealth" in web_agent:
        return "crawl4ai-stealth"

    if "crawl4ai" in web_agent:
        return "crawl4ai"

    # BrowserUse
    if "browseruse_stealth" in web_agent and "bu-1" in model:
        return "browseruse stealth bu1"

    if "browseruse_stealth" in web_agent and "sonnet" in model:
        return "browseruse stealth sonnet"

    if "browseruse" in web_agent and location == "LOCAL" and "bu-1" in model:
        return "browseruse local bu1"

    if "browseruse" in web_agent and location == "LOCAL" and "sonnet" in model:
        return "browseruse local sonnet"

    if "browseruse" in web_agent and location == "CLOUD" and "bu-2" in model:
        return "browseruse cloud bu2"

    if "browseruse" in web_agent and location == "CLOUD" and "sonnet" in model:
        return "browseruse cloud sonnet"

    # ChatGPT
    if "chatgpt" in web_agent:
        return "chatgpt"

    # Skyvern
    if "skyvern" in web_agent and "optimized" in model:
        return "skyvern opti"

    if "skyvern" in web_agent and "gpt" in model:
        return "skyvern gpt"

    return f"{row['Web Agent']} - {row['Model']} - {row['Agent LOCAL/CLOUD']}"


df["TOOL"] = df.apply(build_agent_label, axis=1)

In [4]:
def is_special_prompt(value):

    if str(value).lower() in ["true", "1", "yes"]:
        return True

    return False


df["SP_FLAG"] = df["Special prompt"].apply(is_special_prompt)


## Table 7

This cell reproduces the table 7:

| Tool | NP visits | SP Visits | NoLLM visits |
|---|---:|---:|---:|

Definitions:
- **NP** = normal prompts
- **SP** = special prompts
- **NoLLM** = non-LLM tools/browsers/humans

The final total follows:

Total = NP + SP + NoLLM

In [5]:
nollm_tools = [
    "human",
    "curl",
    "wget",
    "scrapy",
    "selenium",
    "playwright",
    "puppeteer"
]

rows = []

for tool in sorted(df["TOOL"].unique()):

    tool_df = df[df["TOOL"] == tool]

    if tool in nollm_tools:

        np_visits = "–"
        sp_visits = "–"
        nollm_visits = len(tool_df)

    else:

        np_visits = (~tool_df["SP_FLAG"]).sum()
        sp_visits = tool_df["SP_FLAG"].sum()
        nollm_visits = "–"

    rows.append({
        "Tool": tool,
        "NP visits": np_visits,
        "SP Visits": sp_visits,
        "NoLLM visits": nollm_visits
    })

summary = pd.DataFrame(rows)


np_total = (
    summary["NP visits"]
    .replace("–", 0)
    .astype(int)
    .sum()
)

sp_total = (
    summary["SP Visits"]
    .replace("–", 0)
    .astype(int)
    .sum()
)

nollm_total = (
    summary["NoLLM visits"]
    .replace("–", 0)
    .astype(int)
    .sum()
)

grand_total = np_total + sp_total + nollm_total

totals = pd.DataFrame([{
    "Tool": f"Total: {grand_total} = {np_total} + {sp_total} + {nollm_total}",
    "NP visits": "",
    "SP Visits": "",
    "NoLLM visits": ""
}])

final_table = pd.concat([summary, totals], ignore_index=True)

final_table

,Tool,NP visits,SP Visits,NoLLM visits
0,browseruse cloud bu2,50,15,–
1,browseruse cloud sonnet,50,15,–
2,browseruse local bu1,50,15,–
3,browseruse local sonnet,50,15,–
4,browseruse stealth bu1,50,30,–
5,browseruse stealth sonnet,50,30,–
6,chatgpt,50,15,–
7,claude chrome opus,51,16,–
8,claude chrome sonnet,51,6,–
9,crawl4ai,50,0,–


In [6]:
def classify(row):
    blocked = bool(row["BLOCKED"])
    timeout = bool(row["TIMEOUT"])

    if timeout:
        return "TIMEOUT"
    elif blocked:
        return "BLOCKED"
    else:
        return "SUCCESS"

df["RESULT"] = df.apply(classify, axis=1)

In [7]:
def summarize_group(group):
    total = len(group)

    success = (group["RESULT"] == "SUCCESS").sum()
    blocked = (group["RESULT"] == "BLOCKED").sum()
    timeout = (group["RESULT"] == "TIMEOUT").sum()

    if success == total:
        return "TRUE"

    if blocked == total:
        return "FALSE"

    parts = []

    if success > 0:
        parts.append(f"{success}+")

    if blocked > 0:
        parts.append(f"{blocked}b")

    if timeout > 0:
        parts.append(f"{timeout}t")

    if success >= blocked:
        prefix = "TRUE"
    else:
        prefix = "FALSE"

    return prefix + "".join(parts)


matrix = (
    df.groupby(["TOOL", "Defense"])
      .apply(summarize_group)
      .reset_index(name="VALUE")
)

## Table 3

This cell reproduces Table 3:

| Tool | ND | RT | UA | rV3 | Pro | Anubis | TS | CF | A | B |
| ---- | -- | -- | -- | --- | --- | ------ | -- | -- | - | - |

Definitions:

* **ND** = No Defense
* **RT** = Robots.txt
* **UA** = User-Agent Filtering
* **rV3** = Google reCAPTCHA v3
* **Pro** = Prosopo CAPTCHA
* **TS** = Cloudflare Turnstile
* **CF** = Cloudflare Bot Fight Mode + Block AI Bots
* **A** = Open-source defense combination (RT + UA + Pro + Anubis)
* **B** = Cloudflare defense combination (RT + UA + TS + CF)

Values:

* **TRUE** indicates that the tool successfully reached the protected page and completed the assigned task.
* **FALSE** indicates that the tool was blocked or failed to complete the task.
* Entries such as **TRUE4+1t**, **TRUE4+1b**, or **TRUE3+1b1t** provide additional information:

  * The numeric value indicates the number of successful or failed executions.
  * The suffix **+** denotes a challenge "passed" by the bot.
  * The suffix **t** denotes a "time-out".
  * The suffix **b** denotes a "blocked" bot (i.e., a successful defense).
  * Detailed interpretations are available in the paper and accompanying dataset.

The results summarize the effectiveness of each anti-bot mechanism against the evaluated LLM-based bots.

In [8]:
benchmark_table = matrix.pivot(
    index="TOOL",
    columns="Defense",
    values="VALUE"
)

benchmark_table = benchmark_table.fillna("")


# -------------------------------------------------
# RENAME DEFENSE COLUMNS
# -------------------------------------------------

defense_rename = {
    "NONE": "ND",
    "robots.txt": "RT",
    "User-Agent filter": "UA",
    "reCAPTCHA v3": "rV3",
    "Prosopo CAPTCHA": "Pro",
    "Anubis": "Anubis",
    "Cloudflare Turnstile": "TS",
    "Cloudflare anti bot": "CF",
    "1+2+4+5": "A",
    "1+2+6+7": "B"
}

benchmark_table = benchmark_table.rename(columns=defense_rename)


# -------------------------------------------------
# COLUMN ORDER
# -------------------------------------------------

column_order = [
    "ND",
    "RT",
    "UA",
    "rV3",
    "Pro",
    "Anubis",
    "TS",
    "CF",
    "A",
    "B"
]

existing_columns = [
    c for c in column_order
    if c in benchmark_table.columns
]

benchmark_table = benchmark_table[existing_columns]


# -------------------------------------------------
# ROW ORDER
# -------------------------------------------------

row_order = [
    "human",
    "curl",
    "wget",
    "scrapy",
    "selenium",
    "playwright",
    "puppeteer",
    "openclaw sonnet",
    "openclaw opus",
    "claude chrome sonnet",
    "claude chrome opus",
    "crawl4ai",
    "crawl4ai-stealth",
    "crawl4ai-underetected",
    "browseruse local bu1",
    "browseruse local sonnet",
    "browseruse cloud bu2",
    "browseruse cloud sonnet",
    "browseruse stealth bu1",
    "browseruse stealth sonnet",
    "chatgpt",
    "skyvern opti",
    "skyvern gpt"
]

existing_rows = [
    r for r in row_order
    if r in benchmark_table.index
]

benchmark_table = benchmark_table.loc[existing_rows]

benchmark_table


Defense,ND,RT,UA,rV3,Pro,Anubis,TS,CF,A,B
TOOL,,,,,,,,,,
human,TRUE,TRUE,TRUE,TRUE,TRUE,TRUE,TRUE,TRUE,TRUE,TRUE
curl,TRUE,TRUE,TRUE,FALSE,FALSE,FALSE,FALSE,TRUE,FALSE,FALSE
wget,TRUE,TRUE,TRUE,FALSE,FALSE,FALSE,FALSE,TRUE,FALSE,FALSE
scrapy,TRUE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,TRUE,FALSE,FALSE
selenium,TRUE,TRUE,TRUE,TRUE,FALSE,TRUE,FALSE,TRUE,FALSE,FALSE
playwright,TRUE,TRUE,TRUE,TRUE,FALSE,TRUE,FALSE,TRUE,FALSE,FALSE
puppeteer,TRUE,TRUE,TRUE,TRUE,FALSE,TRUE,FALSE,TRUE,FALSE,FALSE1+9b
openclaw sonnet,TRUE11+1t,TRUE3+2t,TRUE,TRUE3+2t,TRUE,TRUE4+1t,TRUE,TRUE4+1b,TRUE3+1b1t,TRUE4+1t
openclaw opus,TRUE,TRUE,TRUE,TRUE,TRUE,TRUE,TRUE,TRUE,TRUE,TRUE
